# Stock Valuation & Volume Dashboard

**What this notebook does**

1. **Values the stock** several ways (peer multiples, a 2-stage DCF, the Graham Number, and optionally analyst targets) and blends them into a **fair value range**.
2. **Checks volume** with six indicators (relative volume, OBV, Chaikin Money Flow, Money Flow Index, VWAP, up/down volume) to see whether buying or selling pressure **supports** the price.
3. **Builds price ranges**: fundamental fair value, the volume *value area*, a 1-month expected move from volatility, and the 52-week range.
4. **Charts everything** interactively, then runs a **live intraday monitor** that refreshes on a timer.

**How to use:** set the ticker and peers in cell 1, then **Runtime → Run all**. Charts are interactive (hover, zoom, double-click to reset).

**Data:** Yahoo Finance through `yfinance`: free, but quotes can be delayed up to ~15 minutes and some fundamentals are occasionally missing. The notebook skips any method it can't compute and tells you why.

*For learning and analysis only, not investment advice.*

In [ ]:
# @title 0. Install packages (run once per session)
%pip install -q yfinance plotly

In [ ]:
# @title 1. Inputs  { display-mode: "form" }
TICKER = "AAPL"  # @param {type:"string"}
# Comparable companies for the multiples valuation (comma-separated). Pick 4–8 true competitors.
PEERS = "MSFT, GOOGL, META, AMZN, NVDA, DELL"  # @param {type:"string"}

# DCF assumptions
EQUITY_RISK_PREMIUM = 0.05  # @param {type:"number"}
TERMINAL_GROWTH = 0.025  # @param {type:"number"}
# Years 1–5 FCF growth. Leave at -1 to auto-estimate from recent growth (capped at 2–15%).
STAGE1_GROWTH = -1.0  # @param {type:"number"}
TAX_RATE = 0.21  # @param {type:"number"}

# Include Wall Street price targets in the composite fair value? (They're always shown for reference.)
INCLUDE_ANALYST_TARGETS = False  # @param {type:"boolean"}

# How much price history to show on the charts
CHART_PERIOD = "1y"  # @param ["3mo", "6mo", "1y", "2y"]

### The measures, in plain English

| Measure | What it asks | Why it matters |
|---|---|---|
| **Peer multiples** (P/E, forward P/E, P/S, P/B, EV/EBITDA) | If this company traded at the same multiple as its competitors, what would it be worth? | Fast, market-based. The range uses the 25th–75th percentile of peers. |
| **DCF (2-stage)** | What are the company's future free cash flows worth today? | Intrinsic value. Range comes from flexing WACC and terminal growth ±0.5%. |
| **Graham Number** | √(22.5 × EPS × book value per share) | A conservative "value investor" ceiling; very low for asset-light growth firms. |
| **Analyst targets** | Where does Wall Street think it goes in 12 months? | Reference point; excluded from the composite unless you turn it on. |
| **Composite fair value** | Median of every method's low, base, and high | Medians keep one odd method from dominating. Price below the range = *undervalued*; above = *overvalued*. |

| Volume indicator | Bullish when… | Bearish when… |
|---|---|---|
| **Relative volume (RVOL)** | Heavy volume (≥1.5× average) on an up day | Heavy volume on a down day |
| **On-Balance Volume (OBV)** | OBV rising with price, or rising while price falls (accumulation) | OBV falling while price rises (divergence) |
| **Chaikin Money Flow (20)** | Above +0.05: closes near the highs on volume | Below −0.05 |
| **Money Flow Index (14)** | Below 20: oversold | Above 80: overbought |
| **20-day VWAP** | Price above the average price paid by recent buyers | Price below it |
| **Up/down volume (20d)** | More volume on up days (ratio > 1.2) | More on down days (< 0.8) |
| **Volume profile** | Shows the **Point of Control** (price with the most volume) and the **value area** where 70% of trading happened. These act as support/resistance. | |

In [ ]:
# @title 2. Setup: imports and data loaders
import math
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def _num(x):
    # Return a finite float, or None if missing / not a number.
    try:
        x = float(x)
        return x if math.isfinite(x) else None
    except (TypeError, ValueError):
        return None


def safe_info(symbol):
    try:
        return yf.Ticker(symbol).info or {}
    except Exception as e:
        print(f"  ! Could not load company info for {symbol}: {e}")
        return {}


def _statement(t, attr):
    try:
        df = getattr(t, attr)
        return df if isinstance(df, pd.DataFrame) else pd.DataFrame()
    except Exception:
        return pd.DataFrame()


def latest(df, row_names):
    # Most recent value of the first matching line item in a yfinance financial statement.
    if df is None or df.empty:
        return None
    for name in row_names:
        if name in df.index:
            row = df.loc[name]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]
            s = pd.to_numeric(row, errors="coerce").dropna()
            try:
                s = s.sort_index(ascending=False)  # newest period first
            except TypeError:
                pass
            if len(s):
                return float(s.iloc[0])
    return None


def load_company(symbol):
    t = yf.Ticker(symbol)
    hist = t.history(period="2y", auto_adjust=False)
    if hist is None or hist.empty:
        raise ValueError(f"No price history found for '{symbol}'. Check the ticker symbol.")
    hist = hist[["Open", "High", "Low", "Close", "Volume"]].dropna()
    info = safe_info(symbol)
    price = (_num(info.get("currentPrice")) or _num(info.get("regularMarketPrice"))
             or float(hist["Close"].iloc[-1]))
    return {"symbol": symbol, "info": info, "hist": hist, "price": price,
            "cf": _statement(t, "cashflow"), "inc": _statement(t, "income_stmt"),
            "bs": _statement(t, "balance_sheet")}


def risk_free_rate(default=0.042):
    # 10-year Treasury yield (^TNX is quoted in percent, e.g. 4.2 = 4.2%).
    try:
        s = yf.Ticker("^TNX").history(period="5d")["Close"].dropna()
        return float(s.iloc[-1]) / 100 if len(s) else default
    except Exception:
        return default


def fundamentals(c):
    i, price = c["info"], c["price"]
    shares = _num(i.get("sharesOutstanding")) or _num(i.get("impliedSharesOutstanding"))
    if not shares and _num(i.get("marketCap")):
        shares = _num(i.get("marketCap")) / price
    revenue = _num(i.get("totalRevenue")) or latest(c["inc"], ["Total Revenue", "Operating Revenue"])

    fcf = latest(c["cf"], ["Free Cash Flow"])
    if fcf is None:
        ocf = latest(c["cf"], ["Operating Cash Flow"])
        capex = latest(c["cf"], ["Capital Expenditure"])
        if ocf is not None and capex is not None:
            fcf = ocf - abs(capex)

    return {
        "name": i.get("longName") or i.get("shortName") or c["symbol"],
        "sector": i.get("sector") or "Unknown",
        "shares": shares,
        "mcap": price * shares if shares else _num(i.get("marketCap")),
        "eps": _num(i.get("trailingEps")),
        "feps": _num(i.get("forwardEps")),
        "bvps": _num(i.get("bookValue")),
        "sps": revenue / shares if revenue and shares else None,
        "ebitda": _num(i.get("ebitda")) or latest(c["inc"], ["EBITDA", "Normalized EBITDA"]),
        "fcf": fcf,
        "debt": _num(i.get("totalDebt")) or latest(c["bs"], ["Total Debt"]) or 0.0,
        "cash": (_num(i.get("totalCash"))
                 or latest(c["bs"], ["Cash Cash Equivalents And Short Term Investments",
                                     "Cash And Cash Equivalents"]) or 0.0),
        "interest": latest(c["inc"], ["Interest Expense", "Interest Expense Non Operating"]),
        "beta": _num(i.get("beta")),
        "growth_hint": [g for g in (_num(i.get("revenueGrowth")), _num(i.get("earningsGrowth")))
                        if g is not None],
    }

print("Setup complete.")

In [ ]:
# @title 3. Valuation models
# label: (yfinance key for the peer's multiple, company metric it applies to, max sensible value)
MULTIPLES = {
    "P/E (trailing)": ("trailingPE", "eps", 150),
    "P/E (forward)": ("forwardPE", "feps", 150),
    "P/S": ("priceToSalesTrailing12Months", "sps", 60),
    "P/B": ("priceToBook", "bvps", 60),
    "EV/EBITDA": ("enterpriseToEbitda", "ebitda", 80),
}
FINANCIAL_SECTORS = {"Financial Services"}  # banks/insurers: FCF and EBITDA don't mean much


def peer_table(peers):
    rows = []
    for p in peers:
        info = safe_info(p)
        row = {"Peer": p}
        for label, (key, _, cap) in MULTIPLES.items():
            v = _num(info.get(key))
            row[label] = v if v is not None and 0 < v <= cap else np.nan  # drop negatives/outliers
        rows.append(row)
        time.sleep(0.25)  # be gentle with Yahoo's rate limits
    return pd.DataFrame(rows).set_index("Peer") if rows else pd.DataFrame()


def multiples_valuation(f, peers_df, skip=()):
    methods = []
    if peers_df.empty:
        return methods
    for label, (_, metric, _) in MULTIPLES.items():
        if label in skip or label not in peers_df:
            continue
        vals = peers_df[label].dropna()
        m = f.get(metric)
        if len(vals) < 2 or not m or m <= 0:
            continue
        q25, q50, q75 = vals.quantile([0.25, 0.50, 0.75])
        if metric == "ebitda":
            if not f["shares"]:
                continue
            prices = [(q * m - f["debt"] + f["cash"]) / f["shares"] for q in (q25, q50, q75)]
        else:
            prices = [q * m for q in (q25, q50, q75)]
        lo, mid, hi = sorted(max(p, 0.0) for p in prices)
        if hi <= 0:
            continue
        methods.append({"Method": f"Peer {label}", "Low": lo, "Mid": mid, "High": hi,
                        "Note": f"peer median {q50:.1f}x across {len(vals)} peers"})
    return methods


def compute_wacc(f, rf, erp, tax):
    beta = f["beta"] if f["beta"] and 0.2 <= f["beta"] <= 3 else 1.0
    cost_equity = rf + beta * erp  # CAPM
    if f["interest"] and f["debt"] > 0:
        cost_debt = min(max(abs(f["interest"]) / f["debt"], rf), rf + 0.06)
    else:
        cost_debt = rf + 0.015
    E, D = (f["mcap"] or 0.0), (f["debt"] or 0.0)
    w_e = E / (E + D) if E + D > 0 else 1.0
    wacc = w_e * cost_equity + (1 - w_e) * cost_debt * (1 - tax)
    return wacc, {"risk-free rate": rf, "beta used": beta, "cost of equity": cost_equity,
                  "pre-tax cost of debt": cost_debt, "equity weight": w_e, "WACC": wacc}


def dcf_price(f, g1, wacc, tg, years=10, fade_start=5):
    # Years 1-5 grow at g1, years 6-10 fade linearly to terminal growth, Gordon growth terminal value.
    if wacc <= tg + 0.005:
        return np.nan
    fcf, pv = f["fcf"], 0.0
    for y in range(1, years + 1):
        g = g1 if y <= fade_start else g1 + (tg - g1) * (y - fade_start) / (years - fade_start)
        fcf *= 1 + g
        pv += fcf / (1 + wacc) ** y
    terminal = fcf * (1 + tg) / (wacc - tg)
    ev = pv + terminal / (1 + wacc) ** years
    return max((ev - f["debt"] + f["cash"]) / f["shares"], 0.0)


def stage1_growth(f, override):
    if override is not None and override >= 0:
        return override, "your input"
    if f["growth_hint"]:
        return float(np.clip(np.mean(f["growth_hint"]), 0.02, 0.15)), "recent revenue/earnings growth, capped 2–15%"
    return 0.06, "default"


def dcf_valuation(f, rf, erp, tg, g1_override, tax):
    out = {"method": None, "grid": None, "assumptions": None, "note": ""}
    if not f["fcf"] or f["fcf"] <= 0 or not f["shares"]:
        out["note"] = "DCF skipped: free cash flow is negative or unavailable."
        return out
    wacc, assumptions = compute_wacc(f, rf, erp, tax)
    g1, g_src = stage1_growth(f, g1_override)
    steps = np.array([-0.01, -0.005, 0.0, 0.005, 0.01])
    waccs, tgs = wacc + steps, tg + steps
    grid = pd.DataFrame([[dcf_price(f, g1, w, t) for t in tgs] for w in waccs],
                        index=[f"{w:.1%}" for w in waccs], columns=[f"{t:.1%}" for t in tgs])
    base = grid.iloc[2, 2]
    core = grid.iloc[1:4, 1:4].values.flatten()
    core = core[np.isfinite(core)]
    if not np.isfinite(base) or len(core) == 0:
        out["note"] = "DCF skipped: WACC is too close to terminal growth."
        return out
    assumptions.update({"stage-1 growth (yrs 1–5)": g1, "terminal growth": tg,
                        "latest free cash flow ($B)": f["fcf"] / 1e9})
    out.update(method={"Method": "DCF (2-stage FCF)", "Low": core.min(), "Mid": base, "High": core.max(),
                       "Note": f"WACC {wacc:.1%}, growth {g1:.1%} ({g_src}), terminal {tg:.1%}"},
               grid=grid, assumptions=assumptions)
    return out


def graham_valuation(f):
    if f["eps"] and f["bvps"] and f["eps"] > 0 and f["bvps"] > 0:
        v = math.sqrt(22.5 * f["eps"] * f["bvps"])
        return {"Method": "Graham Number", "Low": v, "Mid": v, "High": v,
                "Note": "√(22.5 × EPS × book value/share), conservative"}
    return None


def analyst_valuation(info, include):
    lo, mid, hi = (_num(info.get(k)) for k in ("targetLowPrice", "targetMeanPrice", "targetHighPrice"))
    if lo and mid and hi:
        n = info.get("numberOfAnalystOpinions") or "?"
        return {"Method": "Analyst targets", "Low": lo, "Mid": mid, "High": hi,
                "Note": f"{n} analysts" + ("" if include else " (reference only)"),
                "InComposite": include}
    return None


def composite_range(methods, price):
    use = [m for m in methods if m.get("InComposite", True)]
    if not use:
        return None
    lo, mid, hi = sorted(float(np.median([m[k] for m in use])) for k in ("Low", "Mid", "High"))
    verdict = "UNDERVALUED" if price < lo else "OVERVALUED" if price > hi else "FAIRLY VALUED"
    return {"Low": lo, "Mid": mid, "High": hi, "Verdict": verdict,
            "Upside": mid / price - 1, "Methods used": len(use)}

print("Valuation models ready.")

In [ ]:
# @title 4. Volume indicators and price ranges
def add_indicators(df):
    d = df.copy()
    tp = (d["High"] + d["Low"] + d["Close"]) / 3  # typical price
    d["SMA50"] = d["Close"].rolling(50).mean()
    d["SMA200"] = d["Close"].rolling(200).mean()
    d["VWAP20"] = (tp * d["Volume"]).rolling(20).sum() / d["Volume"].rolling(20).sum()

    direction = np.sign(d["Close"].diff()).fillna(0)
    d["OBV"] = (direction * d["Volume"]).cumsum()

    rng = (d["High"] - d["Low"]).replace(0, np.nan)
    mf_mult = ((d["Close"] - d["Low"]) - (d["High"] - d["Close"])) / rng
    mf_vol = mf_mult.fillna(0) * d["Volume"]
    d["CMF20"] = mf_vol.rolling(20).sum() / d["Volume"].rolling(20).sum()

    raw_flow = tp * d["Volume"]
    pos = raw_flow.where(tp > tp.shift(1), 0.0).rolling(14).sum()
    neg = raw_flow.where(tp < tp.shift(1), 0.0).rolling(14).sum()
    d["MFI14"] = 100 - 100 / (1 + pos / neg.replace(0, np.nan))
    d.loc[(neg == 0) & (pos > 0), "MFI14"] = 100.0

    d["VolAvg50"] = d["Volume"].rolling(50).mean()
    d["RVOL"] = d["Volume"] / d["VolAvg50"]
    return d


def volume_profile(df, lookback=126, bins=40, area=0.70):
    d = df.tail(lookback)
    tp = ((d["High"] + d["Low"] + d["Close"]) / 3).values
    edges = np.linspace(d["Low"].min(), d["High"].max(), bins + 1)
    idx = np.clip(np.digitize(tp, edges) - 1, 0, bins - 1)
    vol = np.bincount(idx, weights=d["Volume"].values, minlength=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    poc = int(vol.argmax())
    lo = hi = poc
    acc, total = vol[poc], vol.sum()
    while acc < area * total and (lo > 0 or hi < bins - 1):  # grow outward from the POC
        down = vol[lo - 1] if lo > 0 else -1
        up = vol[hi + 1] if hi < bins - 1 else -1
        if up >= down:
            hi += 1; acc += vol[hi]
        else:
            lo -= 1; acc += vol[lo]
    return {"centers": centers, "volume": vol, "bin": edges[1] - edges[0],
            "poc": centers[poc], "val": edges[lo], "vah": edges[hi + 1], "days": len(d)}


def trend_strength(series, n=20):
    # Slope over the last n days, scaled so ±0.5 or more counts as a real trend.
    y = series.dropna().tail(n).values
    if len(y) < 5 or np.std(y) == 0:
        return 0.0
    slope = np.polyfit(np.arange(len(y)), y, 1)[0]
    return float(slope * len(y) / np.std(y) / 3)


def volume_signals(d, vp):
    last, prev_close = d.iloc[-1], d["Close"].iloc[-2]
    rows = []
    add = lambda name, value, reading, score: rows.append(
        {"Indicator": name, "Value": value, "Reading": reading, "Score": score})

    rv, up_day = last["RVOL"], last["Close"] >= prev_close
    if rv >= 1.5:
        add("Relative volume", f"{rv:.2f}x",
            "Heavy volume on an up day: buyers in control" if up_day else
            "Heavy volume on a down day: sellers in control", 1 if up_day else -1)
    elif rv <= 0.7:
        add("Relative volume", f"{rv:.2f}x", "Light volume: today's move has little conviction", 0)
    else:
        add("Relative volume", f"{rv:.2f}x", "Normal volume", 0)

    pt, ot = trend_strength(d["Close"]), trend_strength(d["OBV"])
    if pt > 0.5 and ot > 0.5:
        r, s = "Price and OBV both rising: volume confirms the uptrend", 1
    elif pt > 0.5 and ot < -0.5:
        r, s = "Bearish divergence: price up but volume flowing out", -1
    elif pt < -0.5 and ot > 0.5:
        r, s = "Bullish divergence: price down but volume accumulating", 1
    elif pt < -0.5 and ot < -0.5:
        r, s = "Price and OBV both falling: volume confirms the downtrend", -1
    else:
        r, s = "No clear 20-day trend in price or OBV", 0
    add("On-Balance Volume trend", f"price {pt:+.1f} / OBV {ot:+.1f}", r, s)

    cmf = last["CMF20"]
    add("Chaikin Money Flow (20)", f"{cmf:+.3f}",
        "Accumulation: closes near highs on volume" if cmf > 0.05 else
        "Distribution: closes near lows on volume" if cmf < -0.05 else "Balanced",
        1 if cmf > 0.05 else -1 if cmf < -0.05 else 0)

    mfi = last["MFI14"]
    add("Money Flow Index (14)", f"{mfi:.0f}",
        "Overbought (>80): buying may be exhausted" if mfi > 80 else
        "Oversold (<20): selling may be exhausted" if mfi < 20 else "Neutral range",
        -1 if mfi > 80 else 1 if mfi < 20 else 0)

    above = last["Close"] > last["VWAP20"]
    add("Price vs 20-day VWAP", f"${last['Close']:.2f} vs ${last['VWAP20']:.2f}",
        "Above VWAP: recent buyers are in profit" if above else "Below VWAP: recent buyers are underwater",
        1 if above else -1)

    w = d.tail(21)
    chg, vol = w["Close"].diff().iloc[1:], w["Volume"].iloc[1:]
    up_v, dn_v = vol[chg > 0].sum(), vol[chg < 0].sum()
    ratio = up_v / dn_v if dn_v else float("inf")
    add("Up/down volume (20d)", f"{ratio:.2f}",
        "More volume on up days" if ratio > 1.2 else "More volume on down days" if ratio < 0.8 else "Even",
        1 if ratio > 1.2 else -1 if ratio < 0.8 else 0)

    px = last["Close"]
    where = ("above the value area: price has moved past where most shares traded" if px > vp["vah"] else
             "below the value area: price is under where most shares traded" if px < vp["val"] else
             "inside the value area: price is at an 'accepted' level")
    add("Volume profile location", f"POC ${vp['poc']:.2f}", f"Price is {where}", 0)

    sig = pd.DataFrame(rows)
    score = int(sig["Score"].sum())
    overall = ("Volume SUPPORTS BUYERS (accumulation)" if score >= 2 else
               "Volume SUPPORTS SELLERS (distribution)" if score <= -2 else "Volume is MIXED / neutral")
    return sig, score, overall


def expected_move(d, days=21, lookback=63):
    r = np.log(d["Close"]).diff().dropna().tail(lookback)
    sigma, p = r.std() * math.sqrt(days), d["Close"].iloc[-1]
    return p * math.exp(-sigma), p * math.exp(sigma), r.std() * math.sqrt(252)


def range_table(comp, vp, em, d, price):
    rows = []
    if comp:
        rows.append(["Fundamental fair value (composite)", comp["Low"], comp["High"],
                     "What the business is worth, per the valuation models"])
    rows.append([f"Volume value area ({vp['days']} days)", vp["val"], vp["vah"],
                 f"Where 70% of shares traded; most-traded price (POC) ${vp['poc']:.2f}"])
    rows.append(["Expected 1-month move (±1σ)", em[0], em[1],
                 f"~68% odds band from {em[2]:.0%} annualized volatility"])
    yr = d.tail(252)
    rows.append(["52-week range", yr["Low"].min(), yr["High"].max(), "Trading extremes over the past year"])
    t = pd.DataFrame(rows, columns=["Range", "Low", "High", "Meaning"])
    pos = (price - t["Low"]) / (t["High"] - t["Low"])
    t["Price position"] = ["below range" if p < 0 else "above range" if p > 1 else f"{p:.0%} of the way up"
                           for p in pos]
    return t


COMBINED = {
    ("UNDERVALUED", 1): "Cheap on fundamentals AND buyers are stepping in: value with confirmation, the strongest combination.",
    ("UNDERVALUED", -1): "Cheap on fundamentals but still being sold. Could be early or a value trap; wait for volume to turn.",
    ("UNDERVALUED", 0): "Cheap on fundamentals, but volume hasn't confirmed. Watch for rising OBV/CMF as a trigger.",
    ("FAIRLY VALUED", 1): "Priced about right, with buying pressure. Moves are more momentum than mispricing.",
    ("FAIRLY VALUED", -1): "Priced about right, with selling pressure. Little valuation cushion if selling continues.",
    ("FAIRLY VALUED", 0): "Priced about right with no strong volume signal. No clear edge either way.",
    ("OVERVALUED", 1): "Expensive, but buyers keep paying up. Momentum can run, but the valuation cushion is thin.",
    ("OVERVALUED", -1): "Expensive AND under distribution: the highest-risk combination.",
    ("OVERVALUED", 0): "Expensive with no strong volume signal. Valuation risk without clear confirmation.",
}

def combined_read(verdict, score):
    bucket = 1 if score >= 2 else -1 if score <= -2 else 0
    return COMBINED.get((verdict, bucket), "Not enough valuation data for a combined read.")

print("Volume and range tools ready.")

In [ ]:
# @title 5. Chart builders
GREEN, RED, BLUE, ORANGE, PURPLE, GRAY, INK = "#16a34a", "#dc2626", "#2563eb", "#ea580c", "#7c3aed", "#6b7280", "#0f172a"
TEMPLATE = "plotly_white"


def add_fair_band(fig, comp, data_lo, data_hi, label_pos="top left", **rc):
    # Shade the fair value range. If it sits far outside the price data, clip it (or just
    # label it) so the candles stay readable instead of being squashed.
    if not comp:
        return
    pad = 0.35 * (data_hi - data_lo)
    win_lo, win_hi = data_lo - pad, data_hi + pad
    label = f"Fair value ${comp['Low']:,.0f}–${comp['High']:,.0f}"
    if comp["High"] < win_lo or comp["Low"] > win_hi:
        arrow = "↑ above" if comp["Low"] > win_hi else "↓ below"
        fig.add_hline(y=win_hi if comp["Low"] > win_hi else win_lo, line_width=0, opacity=0,
                      annotation_text=f"{label} ({arrow} this chart)", annotation_position=label_pos,
                      annotation_font_color=GREEN, **rc)
        return
    y0, y1 = max(comp["Low"], win_lo), min(comp["High"], win_hi)
    clipped = " (extends off chart)" if (y0, y1) != (comp["Low"], comp["High"]) else ""
    fig.add_hrect(y0=y0, y1=y1, fillcolor=GREEN, opacity=0.10, line_width=0,
                  annotation_text=label + clipped, annotation_position=label_pos,
                  annotation_font_color=GREEN, **rc)


def chart_dashboard(d, symbol, comp, vp):
    fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.025,
                        row_heights=[0.46, 0.14, 0.13, 0.13, 0.14],
                        subplot_titles=("Price, moving averages, VWAP and fair value range", "Volume",
                                        "On-Balance Volume", "Chaikin Money Flow (20)", "Money Flow Index (14)"))
    fig.add_trace(go.Candlestick(x=d.index, open=d["Open"], high=d["High"], low=d["Low"], close=d["Close"],
                                 name="Price", increasing_line_color=GREEN, decreasing_line_color=RED), 1, 1)
    for col, color, dash in [("SMA50", BLUE, None), ("SMA200", PURPLE, None), ("VWAP20", ORANGE, "dot")]:
        fig.add_trace(go.Scatter(x=d.index, y=d[col], name=col, line=dict(color=color, width=1.5, dash=dash)), 1, 1)
    add_fair_band(fig, comp, d["Low"].min(), d["High"].max(), row=1, col=1)
    fig.add_hline(y=vp["poc"], line_dash="dash", line_color=GRAY, line_width=1, row=1, col=1,
                  annotation_text=f"POC ${vp['poc']:.2f}", annotation_position="bottom right")

    up = d["Close"] >= d["Open"]
    fig.add_trace(go.Bar(x=d.index, y=d["Volume"], marker_color=np.where(up, GREEN, RED), opacity=0.6,
                         name="Volume", showlegend=False), 2, 1)
    fig.add_trace(go.Scatter(x=d.index, y=d["VolAvg50"], name="50-day avg volume",
                             line=dict(color=INK, width=1.2)), 2, 1)
    fig.add_trace(go.Scatter(x=d.index, y=d["OBV"], name="OBV", line=dict(color=BLUE, width=1.5),
                             showlegend=False), 3, 1)
    fig.add_trace(go.Bar(x=d.index, y=d["CMF20"], marker_color=np.where(d["CMF20"] >= 0, GREEN, RED),
                         name="CMF", showlegend=False), 4, 1)
    fig.add_trace(go.Scatter(x=d.index, y=d["MFI14"], name="MFI", line=dict(color=PURPLE, width=1.5),
                             showlegend=False), 5, 1)
    for lvl in (80, 20):
        fig.add_hline(y=lvl, line_dash="dot", line_color=GRAY, line_width=1, row=5, col=1)
    fig.update_yaxes(range=[0, 100], row=5, col=1)

    fig.update_layout(template=TEMPLATE, height=1080, hovermode="x unified",
                      title=dict(text=f"<b>{symbol}</b> price and volume dashboard", y=0.985),
                      xaxis_rangeslider_visible=False, margin=dict(l=60, r=30, t=120, b=40),
                      legend=dict(orientation="h", yanchor="bottom", y=1.035, x=0, xanchor="left"))
    fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])])
    fig.update_xaxes(rangeselector=dict(x=1, xanchor="right", y=1.035, yanchor="bottom",
                                        buttons=[dict(count=1, label="1M", step="month", stepmode="backward"),
                                                 dict(count=3, label="3M", step="month", stepmode="backward"),
                                                 dict(count=6, label="6M", step="month", stepmode="backward"),
                                                 dict(step="all", label="All")]), row=1, col=1)
    return fig


def chart_volume_profile(vp, price, comp, symbol):
    inside = (vp["centers"] >= vp["val"]) & (vp["centers"] <= vp["vah"])
    fig = go.Figure(go.Bar(x=vp["volume"], y=vp["centers"], orientation="h", width=vp["bin"] * 0.9,
                           marker_color=np.where(inside, BLUE, "#cbd5e1"), name="Volume at price",
                           hovertemplate="$%{y:.2f}: %{x:,.0f} shares<extra></extra>"))
    add_fair_band(fig, comp, min(vp["centers"].min(), price), max(vp["centers"].max(), price), "top right")
    fig.add_hline(y=vp["poc"], line_dash="dash", line_color=GRAY, annotation_text=f"POC ${vp['poc']:.2f}",
                  annotation_position="bottom right")
    fig.add_hline(y=price, line_color=ORANGE, line_width=2.5, annotation_text=f"Price ${price:.2f}",
                  annotation_position="top right")
    fig.update_layout(template=TEMPLATE, height=620, showlegend=False,
                      title=(f"<b>{symbol}</b> volume profile, last {vp['days']} trading days"
                             f"<br><sup>Blue = value area (70% of volume): ${vp['val']:.2f} – ${vp['vah']:.2f}</sup>"),
                      xaxis_title="Shares traded", yaxis_title="Price ($)")
    return fig


def chart_football_field(methods, price, comp, symbol):
    df = pd.DataFrame(methods).sort_values("Mid").reset_index(drop=True)
    in_comp = df.get("InComposite", pd.Series(True, index=df.index)).fillna(True).astype(bool)
    width = (df["High"] - df["Low"]).clip(lower=price * 0.006)
    hover = [f"<b>{m}</b><br>Low ${l:,.2f} · Base ${b:,.2f} · High ${h:,.2f}<br>{n}"
             for m, l, b, h, n in zip(df["Method"], df["Low"], df["Mid"], df["High"], df["Note"])]
    fig = go.Figure()
    fig.add_trace(go.Bar(y=df["Method"], x=width, base=df["Low"], orientation="h", name="Low–high range",
                         marker_color=np.where(in_comp, BLUE, "#cbd5e1"), hovertext=hover,
                         hoverinfo="text", opacity=0.85))
    fig.add_trace(go.Scatter(y=df["Method"], x=df["Mid"], mode="markers", name="Base case",
                             marker=dict(symbol="diamond", size=11, color=INK), hovertext=hover, hoverinfo="text"))
    if comp:
        fig.add_vrect(x0=comp["Low"], x1=comp["High"], fillcolor=GREEN, opacity=0.12, line_width=0,
                      annotation_text="Composite fair value", annotation_position="top left")
    fig.add_vline(x=price, line_color=ORANGE, line_width=2.5, annotation_text=f"Price ${price:.2f}",
                  annotation_position="top right")
    fig.update_layout(template=TEMPLATE, height=170 + 55 * len(df), barmode="overlay",
                      title=f"<b>{symbol}</b> valuation football field<br><sup>Gray = shown for reference, not in composite</sup>",
                      xaxis_title="Implied share price ($)", legend=dict(orientation="h", y=-0.2),
                      margin=dict(l=150, r=40, t=90, b=60))
    return fig


def chart_gauge(price, comp, symbol):
    lo, mid, hi = comp["Low"], comp["Mid"], comp["High"]
    amin, amax = min(lo, price) * 0.8, max(hi, price) * 1.2
    fig = go.Figure(go.Indicator(
        mode="gauge+number+delta", value=price, number={"prefix": "$", "valueformat": ",.2f"},
        delta={"reference": mid, "relative": True, "valueformat": ".1%",
               "increasing": {"color": RED}, "decreasing": {"color": GREEN}},
        title={"text": f"<b>{symbol}: {comp['Verdict']}</b><br><span style='font-size:0.75em;color:{GRAY}'>"
                       f"Price vs. fair value (base ${mid:,.2f}); delta = premium/discount</span>"},
        gauge={"axis": {"range": [amin, amax], "tickprefix": "$"},
               "bar": {"color": INK, "thickness": 0.22},
               "steps": [{"range": [amin, lo], "color": "#bbf7d0"},
                         {"range": [lo, hi], "color": "#fef08a"},
                         {"range": [hi, amax], "color": "#fecaca"}],
               "threshold": {"line": {"color": ORANGE, "width": 4}, "value": price}}))
    fig.update_layout(template=TEMPLATE, height=380, margin=dict(l=40, r=40, t=110, b=20))
    return fig


def chart_dcf_heatmap(grid, price, symbol):
    upside = grid.values / price - 1
    text = [[f"${v:,.0f}<br>{u:+.0%}" if np.isfinite(v) else "n/a" for v, u in zip(rv, ru)]
            for rv, ru in zip(grid.values, upside)]
    fig = go.Figure(go.Heatmap(z=upside, x=grid.columns, y=grid.index, text=text, texttemplate="%{text}",
                               colorscale="RdYlGn", zmid=0, colorbar=dict(title="vs. price", tickformat=".0%"),
                               hovertemplate="WACC %{y}, terminal growth %{x}<br>%{text}<extra></extra>"))
    fig.update_layout(template=TEMPLATE, height=460,
                      title=f"<b>{symbol}</b> DCF sensitivity: value per share vs. current ${price:.2f}",
                      xaxis_title="Terminal growth rate", yaxis_title="WACC", yaxis_autorange="reversed")
    return fig


def show_table(df, score_col=None, money_cols=(), pct_cols=()):
    fmt = {c: "${:,.2f}" for c in money_cols}
    fmt.update({c: "{:+.1%}" for c in pct_cols})
    sty = df.style.format(fmt, na_rep="–").hide(axis="index")
    if score_col:
        color = lambda v: f"color: {GREEN}; font-weight: 600" if v > 0 else (
            f"color: {RED}; font-weight: 600" if v < 0 else f"color: {GRAY}")
        sty = sty.map(color, subset=[score_col])
    display(sty)

print("Chart builders ready.")

In [ ]:
# @title 6. Run the analysis
CHART_DAYS = {"3mo": 63, "6mo": 126, "1y": 252, "2y": 504}
symbol = TICKER.strip().upper()
peers = [p.strip().upper() for p in PEERS.split(",") if p.strip() and p.strip().upper() != symbol]

print(f"Loading {symbol} …")
co = load_company(symbol)
f = fundamentals(co)
price = co["price"]
rf = risk_free_rate()
is_financial = f["sector"] in FINANCIAL_SECTORS

# ── Valuation ──
print(f"Loading {len(peers)} peers for multiples …")
peers_df = peer_table(peers) if peers else pd.DataFrame()
methods = multiples_valuation(f, peers_df, skip={"EV/EBITDA", "P/S"} if is_financial else ())

if is_financial:
    dcf = {"method": None, "grid": None, "assumptions": None,
           "note": "DCF skipped: cash-flow models don't fit banks/insurers; lean on P/B and P/E."}
else:
    g_in = STAGE1_GROWTH if STAGE1_GROWTH >= 0 else None
    dcf = dcf_valuation(f, rf, EQUITY_RISK_PREMIUM, TERMINAL_GROWTH, g_in, TAX_RATE)
if dcf["method"]:
    methods.append(dcf["method"])
for extra in (graham_valuation(f), analyst_valuation(co["info"], INCLUDE_ANALYST_TARGETS)):
    if extra:
        methods.append(extra)
comp = composite_range(methods, price)

# ── Volume and ranges ──
d_all = add_indicators(co["hist"])
vp = volume_profile(d_all)
signals, vscore, voverall = volume_signals(d_all, vp)
em = expected_move(d_all)
ranges = range_table(comp, vp, em, d_all, price)
d = d_all.tail(CHART_DAYS.get(CHART_PERIOD, 252))

# ── Report ──
line = "─" * 78
print(f"\n{line}\n{f['name']} ({symbol})  ·  {f['sector']}  ·  price ${price:,.2f}\n{line}")
if comp:
    print(f"VALUATION : {comp['Verdict']}. Fair value ${comp['Low']:,.2f} – ${comp['High']:,.2f} "
          f"(base ${comp['Mid']:,.2f}, {comp['Upside']:+.1%} vs. price), {comp['Methods used']} methods")
else:
    print("VALUATION : not enough data for any method.")
print(f"VOLUME    : {voverall} (net score {vscore:+d}, scale −6 to +6)")
print(f"READ      : {combined_read(comp['Verdict'] if comp else None, vscore)}")
if dcf["note"]:
    print(f"NOTE      : {dcf['note']}")
print(line)

if methods:
    mt = pd.DataFrame(methods).drop(columns="InComposite", errors="ignore")
    mt["Upside to base"] = mt["Mid"] / price - 1
    print("\nValuation methods")
    show_table(mt[["Method", "Low", "Mid", "High", "Upside to base", "Note"]],
               money_cols=("Low", "Mid", "High"), pct_cols=("Upside to base",))
if not peers_df.empty:
    print("\nPeer multiples (blank = negative, missing, or outlier)")
    display(peers_df.style.format("{:.1f}x", na_rep="–"))
if dcf["assumptions"]:
    print("\nDCF assumptions")
    a = dcf["assumptions"]
    display(pd.DataFrame({"Assumption": list(a),
                          "Value": [f"{v:.2f}" if "beta" in k or "$B" in k else f"{v:.1%}" for k, v in a.items()]})
            .style.hide(axis="index"))
print("\nVolume signals")
show_table(signals, score_col="Score")
print("\nPrice ranges")
show_table(ranges, money_cols=("Low", "High"))

### How to read the charts
- **Gauge:** where today's price sits against fair value. Green zone = below the range (cheap), yellow = inside, red = above (expensive).
- **Football field:** each valuation method's low–high range, with the base case as a diamond. The orange line is today's price. When most bars sit to the right of it, the methods agree the stock is cheap.
- **DCF heatmap:** how much the DCF value moves when you change WACC and terminal growth. A wide spread means the valuation is very assumption-sensitive.
- **Dashboard:** price with 50/200-day averages and 20-day VWAP, the fair value band in green, and the volume indicators underneath. Look for agreement: price rising **with** rising OBV and positive CMF is healthier than price rising on fading volume.
- **Volume profile:** the horizontal bars show where shares actually traded. Prices with heavy volume (the POC and value area) tend to act as support or resistance.

In [ ]:
# @title 7. Charts
if comp:
    chart_gauge(price, comp, symbol).show()
if methods:
    chart_football_field(methods, price, comp, symbol).show()
if dcf["grid"] is not None:
    chart_dcf_heatmap(dcf["grid"], price, symbol).show()
chart_dashboard(d, symbol, comp, vp).show()
chart_volume_profile(vp, price, comp, symbol).show()

### Piotroski F-Score
Nine pass/fail tests on profitability, leverage/liquidity, and operating efficiency, comparing the latest fiscal year to the one before. 8–9 = strong and improving; 0–2 = deteriorating fundamentals (the classic "avoid" or short signal). Pairs well with valuation: a cheap stock with a low F-score is often cheap for a reason.

In [ ]:
# @title 7b. Piotroski F-Score (financial health, 0–9)
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _periods(df):
    # Annual statement with newest year first and empty years dropped.
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return pd.DataFrame()
    df = df.dropna(axis=1, how="all")
    try:
        df = df[sorted(df.columns, reverse=True)]
    except TypeError:
        pass
    return df


def _val(df, names, k):
    # Value of the first matching line item, k years back (0 = latest fiscal year).
    for n in names:
        if n in df.index and df.shape[1] > k:
            row = df.loc[n]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]
            v = pd.to_numeric(row, errors="coerce").iloc[k]
            if pd.notna(v):
                return float(v)
    return None


def _div(a, b):
    return a / b if a is not None and b not in (None, 0) else None


def piotroski_fscore(inc, bs, cf):
    inc, bs, cf = _periods(inc), _periods(bs), _periods(cf)
    if min(inc.shape[1], bs.shape[1], cf.shape[1]) < 2:
        raise ValueError("Need at least two years of annual statements for an F-score.")

    def get(df, names, k):
        return _val(df, names, k)

    NI = ["Net Income", "Net Income Common Stockholders", "Net Income From Continuing Operation Net Minority Interest"]
    REV = ["Total Revenue", "Operating Revenue"]
    TA = ["Total Assets"]
    LTD = ["Long Term Debt", "Long Term Debt And Capital Lease Obligation"]
    CA, CL = ["Current Assets"], ["Current Liabilities"]
    SH = ["Ordinary Shares Number", "Share Issued"]
    CFO = ["Operating Cash Flow", "Cash Flow From Continuing Operating Activities"]

    def assets_begin(k):
        # Beginning-of-year assets = prior year-end; fall back to year-end if history is short.
        return get(bs, TA, k + 1) or get(bs, TA, k)

    def gross_margin(k):
        gp = get(inc, ["Gross Profit"], k)
        rev = get(inc, REV, k)
        if gp is None:
            cogs = get(inc, ["Cost Of Revenue"], k)
            gp = rev - cogs if rev is not None and cogs is not None else None
        return _div(gp, rev)

    def shares(k):
        return get(bs, SH, k) or get(inc, ["Diluted Average Shares", "Basic Average Shares"], k)

    roa0, roa1 = _div(get(inc, NI, 0), assets_begin(0)), _div(get(inc, NI, 1), assets_begin(1))
    cfo0 = get(cf, CFO, 0)
    lev0 = _div(get(bs, LTD, 0) or 0.0, get(bs, TA, 0))
    lev1 = _div(get(bs, LTD, 1) or 0.0, get(bs, TA, 1))
    cr0, cr1 = _div(get(bs, CA, 0), get(bs, CL, 0)), _div(get(bs, CA, 1), get(bs, CL, 1))
    gm0, gm1 = gross_margin(0), gross_margin(1)
    at0, at1 = _div(get(inc, REV, 0), assets_begin(0)), _div(get(inc, REV, 1), assets_begin(1))
    sh0, sh1 = shares(0), shares(1)
    cfo_ta = _div(cfo0, assets_begin(0))

    def test(group, name, ok, now, before, fmt):
        f = (lambda v: "–" if v is None else fmt.format(v))
        return {"Group": group, "Test": name, "Pass": None if ok is None else bool(ok),
                "This year": f(now), "Last year": f(before)}

    both = lambda a, b: a is not None and b is not None
    rows = [
        test("Profitability", "Positive return on assets", None if roa0 is None else roa0 > 0, roa0, roa1, "{:.1%}"),
        test("Profitability", "Positive operating cash flow", None if cfo0 is None else cfo0 > 0,
             None if cfo0 is None else cfo0 / 1e9, None, "{:,.1f} bn"),
        test("Profitability", "ROA improved", (roa0 > roa1) if both(roa0, roa1) else None, roa0, roa1, "{:.1%}"),
        test("Profitability", "Cash flow > earnings (quality)", (cfo_ta > roa0) if both(cfo_ta, roa0) else None,
             cfo_ta, None, "CFO/assets {:.1%}"),
        test("Leverage & liquidity", "Long-term debt ratio didn't rise", (lev0 <= lev1) if both(lev0, lev1) else None,
             lev0, lev1, "{:.1%}"),
        test("Leverage & liquidity", "Current ratio improved", (cr0 > cr1) if both(cr0, cr1) else None, cr0, cr1, "{:.2f}"),
        test("Leverage & liquidity", "No new shares issued", (sh0 <= sh1 * 1.005) if both(sh0, sh1) else None,
             None if sh0 is None else sh0 / 1e6, None if sh1 is None else sh1 / 1e6, "{:,.0f}M"),
        test("Operating efficiency", "Gross margin improved", (gm0 > gm1) if both(gm0, gm1) else None, gm0, gm1, "{:.1%}"),
        test("Operating efficiency", "Asset turnover improved", (at0 > at1) if both(at0, at1) else None, at0, at1, "{:.3f}x"),
    ]
    if roa0 is not None:  # the quality test compares two numbers from the same year
        rows[3]["Last year"] = f"vs ROA {roa0:.1%}"
    rows[1]["Last year"] = ""
    t = pd.DataFrame(rows)
    score, available = int(t["Pass"].eq(True).sum()), int(t["Pass"].notna().sum())
    return score, available, t


F_BANDS = [  # (min score, label, color)
    (8, "STRONG: improving fundamentals", "#15803d"),
    (6, "HEALTHY", "#65a30d"),
    (4, "NEUTRAL / MIXED", "#eab308"),
    (2, "WEAK: deteriorating", "#ea580c"),
    (0, "DISTRESSED: dying-stock warning", "#b91c1c"),
]


def f_band(score_9):
    return next((lbl, c) for lo, lbl, c in F_BANDS if score_9 >= lo)


def chart_fscore(score, available, t, symbol):
    # Scale to 9 when a few tests can't be computed (e.g., banks have no current ratio/gross margin).
    score_9 = score if available == 9 else round(score * 9 / available) if available else 0
    label, color = f_band(score_9)
    fig = make_subplots(rows=2, cols=1, row_heights=[0.42, 0.58], vertical_spacing=0.04,
                        specs=[[{"type": "indicator"}], [{"type": "table"}]])
    fig.add_trace(go.Indicator(
        mode="gauge+number", value=score_9,
        number={"suffix": " / 9", "font": {"size": 46, "color": color}},
        gauge={"axis": {"range": [0, 9], "dtick": 1},
               "bar": {"color": "#0f172a", "thickness": 0.25},
               "steps": [{"range": [0, 2], "color": "#fecaca"}, {"range": [2, 4], "color": "#fed7aa"},
                         {"range": [4, 6], "color": "#fef08a"}, {"range": [6, 8], "color": "#d9f99d"},
                         {"range": [8, 9], "color": "#86efac"}],
               "threshold": {"line": {"color": color, "width": 5}, "value": score_9}}), 1, 1)

    mark = t["Pass"].map({True: "✔ Pass", False: "✘ Fail"}).fillna("– N/A")
    fill = t["Pass"].map({True: "#dcfce7", False: "#fee2e2"}).fillna("#f3f4f6")
    fig.add_trace(go.Table(
        columnwidth=[1.3, 2.4, 0.9, 1.1, 1.1],
        header=dict(values=["<b>Group</b>", "<b>Test</b>", "<b>Result</b>", "<b>This year</b>", "<b>Last year</b>"],
                    fill_color="#0f172a", font=dict(color="white", size=13), align="left", height=30),
        cells=dict(values=[t["Group"], t["Test"], mark, t["This year"], t["Last year"]],
                   fill_color=[fill] * 5, align="left", height=28, font=dict(size=12))), 2, 1)
    note = "" if available == 9 else (f"<br><sup style='color:#6b7280'>{score}/{available} tests passed; "
                                      f"{9 - available} not computable for this company, scaled to 9</sup>")
    fig.update_layout(template="plotly_white", height=800, margin=dict(l=20, r=20, t=110, b=10),
                      title=dict(text=f"<b>{symbol} Piotroski F-Score</b>: "
                                      f"<span style='color:{color}'><b>{label}</b></span>{note}", x=0.5, xanchor="center"))
    return fig, score_9, label


# ── Run it ──
if "co" not in globals():  # standalone use
    _t = yf.Ticker(TICKER.strip().upper())
    co = {"inc": _t.income_stmt, "bs": _t.balance_sheet, "cf": _t.cashflow}
    symbol = TICKER.strip().upper()

try:
    f_score, f_avail, f_table = piotroski_fscore(co["inc"], co["bs"], co["cf"])
    fig_f, f_score_9, f_label = chart_fscore(f_score, f_avail, f_table, symbol)
    print(f"PIOTROSKI : {f_score_9}/9, {f_label}"
          + ("" if f_avail == 9 else f" ({f_score}/{f_avail} computable tests passed)"))
    fig_f.show()
except Exception as e:
    print(f"Piotroski F-score unavailable for {symbol}: {e}")

### Live monitor
Pulls 1-minute bars every `REFRESH_SECONDS` and redraws the intraday chart with session VWAP, prior close, and your fair value band. It stops after `RUN_MINUTES`, or click the **stop** button on the cell. Outside market hours it shows the last session once and stops.

In [ ]:
# @title 8. Live intraday monitor  { display-mode: "form" }
REFRESH_SECONDS = 60  # @param {type:"integer"}
RUN_MINUTES = 15  # @param {type:"integer"}


def live_monitor(symbol, comp, daily, refresh=60, minutes=15):
    refresh = max(int(refresh), 15)  # don't hammer Yahoo
    stop_at = time.time() + minutes * 60
    try:
        while True:
            bars = yf.Ticker(symbol).history(period="5d", interval="1m")
            if bars is None or bars.empty:
                print("No intraday data available right now.")
                return
            session = bars.index[-1].date()
            day = bars[bars.index.date == session].copy()
            tp = (day["High"] + day["Low"] + day["Close"]) / 3
            day["VWAP"] = (tp * day["Volume"]).cumsum() / day["Volume"].cumsum().replace(0, np.nan)
            earlier = daily[daily.index.date < session]
            prev_close = float(earlier["Close"].iloc[-1]) if len(earlier) else float(day["Open"].iloc[0])
            last, vwap = float(day["Close"].iloc[-1]), float(day["VWAP"].iloc[-1])
            age_min = (pd.Timestamp.now(tz=bars.index.tz) - bars.index[-1]).total_seconds() / 60
            market_open = age_min < 10

            if comp:
                status = ("below fair value" if last < comp["Low"] else
                          "above fair value" if last > comp["High"] else "inside fair value")
            else:
                status = "no fair value range"

            fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.75, 0.25], vertical_spacing=0.03)
            fig.add_trace(go.Candlestick(x=day.index, open=day["Open"], high=day["High"], low=day["Low"],
                                         close=day["Close"], name="1-min", increasing_line_color=GREEN,
                                         decreasing_line_color=RED), 1, 1)
            fig.add_trace(go.Scatter(x=day.index, y=day["VWAP"], name="Session VWAP",
                                     line=dict(color=ORANGE, width=2)), 1, 1)
            fig.add_hline(y=prev_close, line_dash="dot", line_color=GRAY, row=1, col=1,
                          annotation_text=f"Prev close ${prev_close:.2f}", annotation_position="bottom left",
                          annotation_bgcolor="rgba(255,255,255,0.8)")
            lo_d, hi_d = min(day["Low"].min(), prev_close), max(day["High"].max(), prev_close)
            add_fair_band(fig, comp, lo_d, hi_d, "top right", row=1, col=1)
            fig.add_trace(go.Bar(x=day.index, y=day["Volume"],
                                 marker_color=np.where(day["Close"] >= day["Open"], GREEN, RED),
                                 name="Volume", opacity=0.6), 2, 1)
            stamp = datetime.now().strftime("%H:%M:%S")
            fig.update_layout(
                template=TEMPLATE, height=620, xaxis_rangeslider_visible=False, showlegend=False,
                title=(f"<b>{symbol}</b> ${last:,.2f} ({last / prev_close - 1:+.2%})  ·  "
                       f"{'above' if last >= vwap else 'below'} VWAP ${vwap:,.2f}  ·  {status}"
                       f"<br><sup>{'Live' if market_open else 'Market closed, last session'} · "
                       f"refreshed {stamp} · data may be delayed</sup>"))
            clear_output(wait=True)
            fig.show()

            if not market_open:
                print("Market is closed, showing the last session. Re-run during market hours for live updates.")
                return
            if time.time() + refresh > stop_at:
                print(f"Monitor finished after {minutes} minutes. Re-run this cell to restart.")
                return
            time.sleep(refresh)
    except KeyboardInterrupt:
        print("Live monitor stopped.")


live_monitor(symbol, comp, co["hist"], REFRESH_SECONDS, RUN_MINUTES)

### Ideas to extend it
- Log each run's verdict, fair value, and volume score to a Google Sheet, then check six months later whether the signals were right. That's a backtest, and a strong talking point.
- Add a dividend discount model for dividend payers, or a residual income model for banks.
- Run a watchlist through the same functions and rank names by discount to fair value plus volume score.